1. Lendo os dados

In [1]:
# lendo os dados de machado.txt.
# ferramentas: with (não precisa de fclose), encoding (transforma bytes em caracteres).
with open('machado.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
# limpeza: remove marcações (* e _) que são erros de transcrição do texto
text = text.replace('_', '').replace('*', '')

In [3]:
print ("tamanho do dataset em caracteres: ", len(text))

tamanho do dataset em caracteres:  1202856


In [4]:
# primeiros 1000 caracteres:
print (text[0:1000])


DOM CASMURRO

POR

MACHADO DE ASSIS

DA ACADEMIA BRAZILEIRA

H. GARNIER, LIVREIRO-EDITOR

RUA MOREIRA CEZAR, 71

RIO DE JANEIRO

6, RUE DES SAINTS-PÈRES, 6

PARIZ




I

Do titulo.

Uma noite destas, vindo da cidade para o Engenho Novo, encontrei no
trem da Central um rapaz aqui do bairro, que eu conheço de vista e
de chapéo. Comprimentou-me, sentou-se ao pé de mim, falou da lua e
dos ministros, e acabou recitando-me versos. A viagem era curta, e os
versos póde ser que não fossem inteiramente maus. Succedeu, porém, que
como eu estava cançado, fechei os olhos tres ou quatro vezes; tanto
bastou para que elle interrompesse a leitura e mettesse os versos no
bolso.

--Continue, disse eu accordando.

--Já acabei, murmurou elle.

--São muito bonitos.

Vi-lhe fazer um gesto para tiral-os outra vez do bolso, mas não passou
do gesto; estava amuado. No dia seguinte entrou a dizer de mim nomes
feios, e acabou alcunhando-me Dom Casmurro. Os visinhos, que não
gostam dos meus habitos reclusos e cala

2. Vocabulário e Tokenização

In [5]:
# todos os caracteres do texto
chars = sorted (list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"$&'()+,-.0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz«°»ÀÁÇÈÉÍÓàáâãæçèéêíîñòóôõúû
107


In [6]:
# tokenizar: converter o texto em tokens
# enumerate: percorre a lista chars e devolve posição (i) e item/caractere (ch).
# stoi: string to integer - string: chave; inteiro: valor.
#itos: integer to string - inteiro: chave; string: valor.
stoi = { ch:i for i,ch in enumerate(chars) } 
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # texto vira lista de números.
decode = lambda l: ''.join([itos[i] for i in l]) # lista de números vira texto.

print(encode("olhos de ressaca"))
print(decode(encode("olhos de ressaca")))

[67, 64, 60, 67, 71, 1, 56, 57, 1, 70, 57, 71, 71, 53, 55, 53]
olhos de ressaca


3. Codificando o texto em tensor

In [7]:
import torch # biblioteca que constrói e cria a rede neural.
# encode(text): transforma os livros numa lista de números.
# torch transforma a lista em um tensor (guarda números organizados em uma ou mais dimensões).
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # mostra os 1000 primeiros números.


/Users/sofiaparrode/Documents/nanoMachado/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1202856]) torch.int64
tensor([  0,  30,  41,  39,   1,  29,  27,  45,  39,  47,  44,  44,  41,   0,
          0,  42,  41,  44,   0,   0,  39,  27,  29,  34,  27,  30,  41,   1,
         30,  31,   1,  27,  45,  45,  35,  45,   0,   0,  30,  27,   1,  27,
         29,  27,  30,  31,  39,  35,  27,   1,  28,  44,  27,  52,  35,  38,
         31,  35,  44,  27,   0,   0,  34,  12,   1,  33,  27,  44,  40,  35,
         31,  44,  10,   1,  38,  35,  48,  44,  31,  35,  44,  41,  11,  31,
         30,  35,  46,  41,  44,   0,   0,  44,  47,  27,   1,  39,  41,  44,
         31,  35,  44,  27,   1,  29,  31,  52,  27,  44,  10,   1,  20,  14,
          0,   0,  44,  35,  41,   1,  30,  31,   1,  36,  27,  40,  31,  35,
         44,  41,   0,   0,  19,  10,   1,  44,  47,  31,   1,  30,  31,  45,
          1,  45,  27,  35,  40,  46,  45,  11,  42,  85,  44,  31,  45,  10,
          1,  19,   0,   0,  42,  27,  44,  35,  52,   0,   0,   0,   0,   0,
         35,   0,   0,  30,  6

4. Separando treino e validação

In [8]:
n = int (0.9*len(data)) # calcula 90% do tamanho do texto
train_data = data[:n] # os primeiros 90% (treino).
val_data = data[n:] # os últimos 10% (validação).

5. Montando os batches
batch:
- diversos pedaços de texto usados para treinar o transformador para que ele possa prever o próximo caractere, tudo ao mesmo tempo.

In [9]:
block_size = 8 # número máximo de caracteres que o modelo olha de uma vez para prever o próximo.
train_data[:block_size+1] # o último caractere sempre é o que será previsto.

tensor([ 0, 30, 41, 39,  1, 29, 27, 45, 39])

In [10]:
x = train_data[:block_size] # entradas.
y = train_data[1:block_size+1] # caractere que será previsto.
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([0]) the target: 30
when input is tensor([ 0, 30]) the target: 41
when input is tensor([ 0, 30, 41]) the target: 39
when input is tensor([ 0, 30, 41, 39]) the target: 1
when input is tensor([ 0, 30, 41, 39,  1]) the target: 29
when input is tensor([ 0, 30, 41, 39,  1, 29]) the target: 27
when input is tensor([ 0, 30, 41, 39,  1, 29, 27]) the target: 45
when input is tensor([ 0, 30, 41, 39,  1, 29, 27, 45]) the target: 39


In [11]:
torch.manual_seed(1337) # fixa seed do gerador de números aleatórios, mas saem iguais toda vez que roda.
batch_size = 4 # 4 trechos diferentes do texto.
block_size = 8 # quantidade de caracteres para cada trecho.
def get_batch(split):
    data = train_data if split == 'train' else val_data
    # sorteia posições iniciais aleatórias, garantindo que cada trecho caiba no texto.
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # recorta, a partir de cada posição sorteada, um trecho de block_size caracteres (entradas).
    x = torch.stack([data[i:i+block_size]for i in ix])
    # recorta os mesmos trechos deslocados em uma posição (alvos: o próximo caractere de cada entrada).
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
# pega um batch sorteado dos dados de treino.
xb, yb = get_batch('train')
# exibe o formato e o conteúdo do batch sorteado.
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

# exibe, para cada trecho do batch, os pares (contexto, alvo) que serão usados no treinamento.
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")
    

inputs:
torch.Size([4, 8])
tensor([[57, 70, 67,  2,  1, 57, 76, 55],
        [70, 53,  1, 69, 73, 57,  1, 67],
        [57, 78,  1, 60, 67, 66, 70, 53],
        [65, 53, 59, 70, 53, 10,  1, 57]])
targets:
torch.Size([4, 8])
tensor([[70, 67,  2,  1, 57, 76, 55, 64],
        [53,  1, 69, 73, 57,  1, 67, 73],
        [78,  1, 60, 67, 66, 70, 53,  1],
        [53, 59, 70, 53, 10,  1, 57, 71]])
----
when input is [57] the target: 70
when input is [57, 70] the target: 67
when input is [57, 70, 67] the target: 2
when input is [57, 70, 67, 2] the target: 1
when input is [57, 70, 67, 2, 1] the target: 57
when input is [57, 70, 67, 2, 1, 57] the target: 76
when input is [57, 70, 67, 2, 1, 57, 76] the target: 55
when input is [57, 70, 67, 2, 1, 57, 76, 55] the target: 64
when input is [70] the target: 53
when input is [70, 53] the target: 1
when input is [70, 53, 1] the target: 69
when input is [70, 53, 1, 69] the target: 73
when input is [70, 53, 1, 69, 73] the target: 57
when input is [70, 53, 

6. Modelo Bigrama
-> rede neural.

In [12]:
import torch
import torch.nn as nn # importa o módulo de redes neurais.
from torch.nn import functional as F # importa funções prontas.
torch.manual_seed(1337)

# define o modelo bigrama, herdando de nn.Module (classe base de todas as redes no PyTorch).
class BigramLanguageModel (nn.Module):

    # construtor.
    def __init__(self, vocab_size):
        # inicializa a classe-mãe nn.Module (necessária para registrar os pesos).
        super().__init__()
        # tabela de pesos vocab_size x vocab_size: cada linha guarda as pontuações
        # do próximo caractere, dado o caractere atual.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # define como o modelo transforma a entrada em previsões.
    def forward(self, idx, targets):
        # para cada caractere de idx, busca sua linha na tabela (logits: pontuações - podem ser negativas - ainda não são probabilidades).
        logits = self.token_embedding_table(idx)
        #retorna as pontuações, com formato (batch_size, block_size, vocab_size).
        return logits

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)
print(out.shape) # mostra o formato dos logits.



torch.Size([4, 8, 107])
